In [1]:
import pandas as pd
from tqdm import trange
print("imports OK")

imports OK


# SAM-SEAD Imperfect Information Extensive Form Game

## Scenario
A **Blue SEAD aircraft** is approaching a **Red SAM battery**.
This is a 2-level sequential game with imperfect information.

```
Level 1 — SAM (Red)  decides first : Radar ON  or Radar OFF
Level 2 — SEAD (Blue) decides second: Attack   or Hold
           SEAD cannot observe the SAM radar state — both Level-2
           nodes are merged into a single infoset.
```

## Payoff Table (Blue / Red, zero-sum)

| SAM state  | SEAD action | Outcome                                        | Blue | Red  |
|------------|-------------|------------------------------------------------|------|------|
| Radar ON   | Attack      | SEAD detects emission, fires ARM, destroys SAM | +4   | −4   |
| Radar ON   | Hold        | SAM active, tracks & engages Blue aircraft     | −5   | +5   |
| Radar OFF  | Attack      | SEAD fires blind, misses, wastes ordnance      | −2   | +2   |
| Radar OFF  | Hold        | SAM silent, SEAD holds — tense standoff        |  0   |  0   |

In [2]:
game_content = """node / player 1 actions on off
node /P1:on  player 2 actions attack hold
node /P1:off player 2 actions attack hold
node /P1:on/P2:attack  leaf payoffs 1=-4 2=4
node /P1:on/P2:hold    leaf payoffs 1=5  2=-5
node /P1:off/P2:attack leaf payoffs 1=2  2=-2
node /P1:off/P2:hold   leaf payoffs 1=0  2=0
infoset pl2_0__ nodes /P1:on /P1:off
"""
with open("sam_sead.game", "w") as f:
    f.write(game_content)
print("file written OK")

file written OK


In [ ]:
import LiteEFG as efg

# Write the game we KNOW works
known_good = """node / player 1 actions H T
node /P1:H player 2 actions H T
node /P1:T player 2 actions H T
node /P1:H/P2:H leaf payoffs 1=-1 2=1
node /P1:H/P2:T leaf payoffs 1=1 2=-1
node /P1:T/P2:H leaf payoffs 1=1 2=-1
node /P1:T/P2:T leaf payoffs 1=-1 2=1
infoset pl2_0__ nodes /P1:H /P1:T
"""
with open("known_good.game", "w") as f:
    f.write(known_good)

env = efg.FileEnv("known_good.game", traverse_type="Enumerate")
print("known_good FileEnv OK")

In [3]:
env2 = efg.FileEnv("sam_sead.game", traverse_type="Enumerate")
print("sam_sead FileEnv OK")

NameError: name 'efg' is not defined

In [ ]:
# ================================================================
# CELL 2 — Run CFR
# ================================================================

NUM_ITER   = 10000
PRINT_FREQ = 1000
history    = []
best_exp   = 1e9

for i in trange(NUM_ITER, desc="CFR (SAM-SEAD)"):
    alg.update_graph(env)
    env.update_strategy(alg.current_strategy(), update_best=False)

    if i % PRINT_FREQ == 0 or i == NUM_ITER - 1:
        explo = sum(env.exploitability(alg.current_strategy(), "avg-iterate"))
        best_exp = min(best_exp, explo)
        history.append({
            "iteration": i,
            "exploitability": round(explo, 8),
            "best": round(best_exp, 8)
        })

print("\nDone.")

In [ ]:
# ================================================================
# CELL 3 — Exploitability convergence table
# ================================================================

print("--- Exploitability convergence ---")
print(pd.DataFrame(history).to_string(index=False))

In [ ]:
# ================================================================
# CELL 4 — Nash Equilibrium mixed strategies + expected game value
# ================================================================

player_labels = {
    1: {"name": "SAM  (Red)",  "actions": ["Radar ON", "Radar OFF"]},
    2: {"name": "SEAD (Blue)", "actions": ["Attack",   "Hold"]},
}

print("=" * 60)
nash = {}   # store probs for expected value calculation
for player_id in range(1, 3):
    info = player_labels[player_id]
    print(f"\nPlayer {player_id} — {info['name']}  |  Nash Equilibrium mixed strategy:")
    pairs = env.get_strategy(player_id, alg.current_strategy(), "avg-iterate")
    nash[player_id] = pairs
    for infoset_name, probs in pairs:
        print(f"  Infoset : {infoset_name}")
        for action, p in zip(info['actions'], probs):
            bar = '█' * int(p * 30)
            print(f"    {action:12s}: {p:.4f}  ({p*100:.1f}%)  {bar}")
print("=" * 60)

# Extract Nash probabilities
p_on  = nash[1][0][1][0]   # P(SAM Radar ON)
q_att = nash[2][0][1][0]   # P(SEAD Attack)

# Expected game value
V_blue = (p_on       * q_att       *  4    # ON  + Attack
        + p_on       * (1 - q_att) * (-5)  # ON  + Hold
        + (1 - p_on) * q_att       * (-2)  # OFF + Attack
        + (1 - p_on) * (1 - q_att) *  0)   # OFF + Hold

print(f"""
Nash Equilibrium Summary
------------------------
  P(SAM Radar ON)    = {p_on:.4f}  ({p_on*100:.1f}%)
  P(SAM Radar OFF)   = {1-p_on:.4f}  ({(1-p_on)*100:.1f}%)

  P(SEAD Attack)     = {q_att:.4f}  ({q_att*100:.1f}%)
  P(SEAD Hold)       = {1-q_att:.4f}  ({(1-q_att)*100:.1f}%)

  Expected value — Blue (SEAD) : {V_blue:+.4f}
  Expected value — Red  (SAM)  : {-V_blue:+.4f}

Interpretation
--------------
  At Nash, both players are indifferent between their actions.
  SAM mixes ON/OFF to prevent SEAD from always attacking or always holding.
  SEAD mixes Attack/Hold to prevent SAM from always emitting or always staying dark.
  The expected value tells us which side has the strategic advantage at equilibrium.
""")